<a href="https://colab.research.google.com/github/algroznykh/closed_form_nca/blob/main/stream_tool_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
Closed-Form NCA with RK4 Self-Organizing Inference
Live inference during training (Colab Decoupled Dual-Channel Dashboard)
"""

import torch, torch.nn as nn, torch.nn.functional as F
import time, cv2, io, numpy as np, traceback
import ipywidgets as widgets
from IPython.display import display, HTML
from collections import deque
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

# Colab-specific interface module
from google.colab import output

torch.backends.cudnn.benchmark = True
device = 'cuda' if torch.cuda.is_available() else 'cpu'

try:
    import clip
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "git+https://github.com/openai/CLIP.git"])
    import clip

clip_model, _ = clip.load('RN101', device=device, jit=False)
clip_model.eval().float()
for p in clip_model.parameters(): p.requires_grad_(False)

CLIP_MEAN = torch.tensor([0.4814, 0.4578, 0.4082], device=device).view(1,3,1,1)
CLIP_STD  = torch.tensor([0.2686, 0.2613, 0.2758], device=device).view(1,3,1,1)
def clip_norm(x): return (x - CLIP_MEAN) / CLIP_STD

def random_crops(img, n=2, size=224):
    B, C, H, W = img.shape
    scales = torch.empty(n, device=img.device).uniform_(0.7, 1.2)
    tx = torch.empty(n, device=img.device).uniform_(-0.1, 0.1)
    ty = torch.empty(n, device=img.device).uniform_(-0.1, 0.1)
    flip = (torch.rand(n, device=img.device) < 0.5).float() * 2 - 1

    theta = torch.zeros(n, 2, 3, device=img.device)
    theta[:,0,0] = scales * flip; theta[:,1,1] = scales
    theta[:,0,2] = tx; theta[:,1,2] = ty

    grid = F.affine_grid(theta, (n, C, size, size), align_corners=False)
    expanded = img.unsqueeze(1).expand(-1, n, -1, -1, -1).reshape(B * n, C, H, W)
    return F.grid_sample(expanded, grid.repeat(B, 1, 1, 1), padding_mode='reflection', align_corners=False)

def get_circular_noise(B, C, size, device):
    noise = torch.randn(B, C, size, size, device=device)
    fy = torch.fft.fftfreq(size, device=device).view(-1, 1)
    fx = torch.fft.fftfreq(size, device=device).view(1, -1)
    mask = torch.exp(-100.0 * (fx**2 + fy**2))
    smooth = torch.fft.ifft2(torch.fft.fft2(noise) * mask).real
    smooth = (smooth - smooth.mean(dim=(2,3), keepdim=True)) / (smooth.std(dim=(2,3), keepdim=True) + 1e-5)
    return smooth * 0.5

get_noise = get_circular_noise

_act = {}
def get_hook(name):
    def _hook(mod, inp, out): _act[name] = out
    return _hook

for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
    dict(clip_model.visual.named_modules())[layer_name].register_forward_hook(get_hook(layer_name))


class ClosedFormNCA(nn.Module):
    def __init__(self, in_ch=12, latent_ch=48, kernel_size=5):
        super().__init__()
        self.C = latent_ch
        self.k_size = kernel_size
        self.shift = kernel_size // 2

        self.lift = nn.Sequential(nn.Conv2d(in_ch, latent_ch, 1), nn.GELU(),
                                  nn.Conv2d(latent_ch, latent_ch, 1), nn.GELU(),
                                  nn.Conv2d(latent_ch, latent_ch, 1))

        self.project = nn.Sequential(nn.Conv2d(latent_ch, latent_ch, 1), nn.GELU(),
                                     nn.Conv2d(latent_ch, latent_ch, 1), nn.GELU(),
                                     nn.Conv2d(latent_ch, 3, 1), nn.Sigmoid())

        self.U_raw = nn.Parameter(torch.randn(latent_ch, latent_ch) * 0.1)
        self.spatial_kernel = nn.Parameter(torch.randn(latent_ch, 1, kernel_size, kernel_size) * 0.05)
        self.damping_raw = nn.Parameter(torch.zeros(latent_ch) - 2.0)

    def forward(self, t, x0):
        z0 = self.lift(x0)
        orig_dtype = z0.dtype

        Z0 = torch.fft.fft2(z0.to(torch.float32))

        U = torch.matrix_exp(self.U_raw.to(torch.float32) - self.U_raw.T.to(torch.float32))
        U_complex = U.to(torch.complex64)

        k = self.spatial_kernel.to(torch.float32)
        pad_w, pad_h = x0.shape[3] - self.k_size, x0.shape[2] - self.k_size
        k_pad = torch.roll(F.pad(k, (0, pad_w, 0, pad_h)), (-self.shift, -self.shift), (2, 3))
        g_hat = torch.fft.fft2(k_pad).squeeze(1)

        decay = F.softplus(self.damping_raw.to(torch.float32)).view(1, -1, 1, 1) * 0.0
        g_stable = (g_hat.real - g_hat.real.amax(dim=(1, 2), keepdim=True) - decay) + 1j * g_hat.imag

        Z_rot = torch.einsum('cd, bdhw -> bchw', U_complex.T, Z0)
        Z_t = Z_rot * torch.exp(t.to(torch.float32) * g_stable)
        Z_final = torch.einsum('cd, bdhw -> bchw', U_complex, Z_t)

        xt = torch.fft.ifft2(Z_final).real.to(orig_dtype)
        return xt, self.project(xt)

    def step_latent_rk4(self, z, dt=0.05):
        orig_dtype = z.dtype
        z_f32 = z.to(torch.float32)

        k = self.spatial_kernel.to(torch.float32)
        pad_w, pad_h = z.shape[3] - self.k_size, z.shape[2] - self.k_size
        k_pad = torch.roll(F.pad(k, (0, pad_w, 0, pad_h)), (-self.shift, -self.shift), (2, 3))

        max_real = torch.fft.fft2(k_pad).squeeze(1).real.amax(dim=(1, 2)).view(self.C)
        decay = F.softplus(self.damping_raw.to(torch.float32)).view(self.C) * 0.0

        k_stable = k.clone()
        k_stable[:, 0, self.shift, self.shift] -= (max_real + decay)

        k_conv = torch.flip(k_stable, [2, 3])
        U = torch.matrix_exp(self.U_raw.to(torch.float32) - self.U_raw.T.to(torch.float32))

        def dynamics(x):
            x_rot = F.pad(torch.einsum('cd, bdhw -> bchw', U.T, x),
                          (self.shift, self.shift, self.shift, self.shift), mode='circular')
            dx_rot = F.conv2d(x_rot, k_conv, groups=self.C)
            return torch.einsum('cd, bdhw -> bchw', U, dx_rot)

        k1 = dynamics(z_f32)
        k2 = dynamics(z_f32 + 0.5 * dt * k1)
        k3 = dynamics(z_f32 + 0.5 * dt * k2)
        k4 = dynamics(z_f32 + dt * k3)

        return (z_f32 + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)).to(orig_dtype)


# Multi-line loss tracking
loss_total_log = []
loss_feature_log = []
loss_l2_log = []
loss_tv_log = []
best_loss = float('inf')
current_title = "Targeting layer2 C42"

# UI controls definition
layer_dropdown = widgets.Dropdown(options=['layer1', 'layer2', 'layer3', 'layer4'], value='layer2', description='Layer:', layout=widgets.Layout(width='180px'))
channel_slider = widgets.IntSlider(min=0, max=511, value=42, description='Channel:', layout=widgets.Layout(width='340px'))
dd_toggle = widgets.Checkbox(value=False, description='DeepDream', layout=widgets.Layout(width='120px'))
btn_restart_train = widgets.Button(description='💣 Hard Reset Model', button_style='danger', icon='bomb', layout=widgets.Layout(width='180px'))

# Viewport Scaler
zoom_slider = widgets.IntSlider(min=256, max=1024, value=512, step=64, description='Scale Viewport:', layout=widgets.Layout(width='300px'))

# Programmatic Video Recording
btn_record = widgets.ToggleButton(description='🔴 Record Video', button_style='info', icon='video-camera', layout=widgets.Layout(width='150px'))
recorded_frames = []

def update_channel_bounds(*args):
    ch_map = {'layer1': 256, 'layer2': 512, 'layer3': 1024, 'layer4': 2048}
    channel_slider.max = ch_map[layer_dropdown.value] - 1
layer_dropdown.observe(update_channel_bounds, 'value')

def update_dd_mode(*args):
    channel_slider.disabled = dd_toggle.value
dd_toggle.observe(update_dd_mode, 'value')

def reset_graph(*args):
    global loss_total_log, loss_feature_log, loss_l2_log, loss_tv_log, best_loss
    loss_total_log.clear()
    loss_feature_log.clear()
    loss_l2_log.clear()
    loss_tv_log.clear()
    best_loss = float('inf')

layer_dropdown.observe(reset_graph, 'value')
channel_slider.observe(reset_graph, 'value')
dd_toggle.observe(reset_graph, 'value')

class InteractiveDOR:
    def __init__(self, shader_func, reset_cb=None, res=128):
        self.shader_func, self.reset_cb = shader_func, reset_cb
        self.is_running = True
        self.is_paused = False
        self.t = self._pause_offset = 0.0
        self._unpause_wall = time.time()

        # Output viewport
        self.img_widget = widgets.Image(value=cv2.imencode('.jpg', np.zeros((res, res*2, 3), np.uint8))[1].tobytes(), format='jpeg')
        self.img_widget.add_class("colab-nca-viewport")
        self.img_widget.layout.width = "512px"
        self.img_widget.layout.height = "256px"

        self.fps_label = widgets.Label(value='Ready')
        self.legend_label = widgets.HTML(value=f"<div style='display:flex; width:100%; text-align:center; font-family:sans-serif; font-size:14px; font-weight:bold; color:#ccc; background:#222; padding:6px 0; border-radius:4px 4px 0 0;'><div style='flex:1;'>⚡ Closed-Form O(1)</div><div style='flex:1;'>🦠 RK4 Cellular Automaton</div></div>")

        # Controls row
        self.btn_toggle = widgets.ToggleButton(value=True, icon='stop', button_style='danger', layout=widgets.Layout(width='32px'))
        self.btn_toggle.observe(self._on_toggle, names='value')
        self.btn_pause = widgets.ToggleButton(icon='pause', button_style='warning', layout=widgets.Layout(width='32px'))
        self.btn_pause.observe(self._on_pause, names='value')
        self.btn_reset = widgets.Button(icon='refresh', button_style='info', layout=widgets.Layout(width='32px'))
        self.btn_reset.on_click(lambda _: self.reset_time())

        # Sub-container layout
        self.ui = widgets.VBox([
            self.legend_label,
            self.img_widget,
            widgets.HBox([self.btn_toggle, self.btn_pause, self.btn_reset, zoom_slider, btn_record, self.fps_label], layout=widgets.Layout(align_items='center', margin='5px 0'))
        ], layout=widgets.Layout(border='1px solid #333', padding='10px', border_radius='4px', background_color='#1a1a1a'))

    def _on_toggle(self, change):
        self.is_running = change['new']
        if self.is_running:
            self.btn_toggle.icon = 'stop'
            self.btn_toggle.button_style = 'danger'
            self._unpause_wall = time.time()
        else:
            self.btn_toggle.icon = 'play'
            self.btn_toggle.button_style = 'success'

    def _on_pause(self, change):
        self.is_paused = change['new']
        if self.is_paused:
            self._pause_offset = self.t
            self.btn_pause.icon = 'play'
        else:
            self._unpause_wall = time.time()
            self.btn_pause.icon = 'pause'

    def reset_time(self):
        self.t = self._pause_offset = 0.0; self._unpause_wall = time.time()
        if self.reset_cb: self.reset_cb()

def handle_zoom(change):
    new_w = change['new']
    dor.img_widget.layout.width = f"{new_w}px"
    dor.img_widget.layout.height = f"{new_w // 2}px"
zoom_slider.observe(handle_zoom, 'value')

# Video recording trigger callback
def handle_record_toggle(change):
    global recorded_frames
    if change['new']:
        recorded_frames = []
        btn_record.description = '⏹️ Stop & Download'
        btn_record.button_style = 'danger'
    else:
        btn_record.description = '⏳ Processing...'
        btn_record.disabled = True
        if len(recorded_frames) > 0:
            try:
                h, w, c = recorded_frames[0].shape
                filename = 'nca_simulation.mp4'
                # Encode via standard mp4v codec
                fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                out_video = cv2.VideoWriter(filename, fourcc, 30.0, (w, h))
                for frame in recorded_frames:
                    out_video.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
                out_video.release()

                # Programmatically request file download via browser context
                from google.colab import files
                files.download(filename)
                stats_label.value = f"💾 Recording exported successfully."
            except Exception as ev:
                print("Video recording encoding error:", ev)
        btn_record.description = '🔴 Record Video'
        btn_record.button_style = 'info'
        btn_record.disabled = False
        recorded_frames = []
btn_record.observe(handle_record_toggle, 'value')

# Model init (Eager Mode)
ca_model = ClosedFormNCA(in_ch=12, latent_ch=48, kernel_size=5).to(device)
optimizer = torch.optim.AdamW(ca_model.parameters(), lr=4e-3, weight_decay=1e-4)
scaler = torch.amp.GradScaler('cuda')

# Loss Canvas Template
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.text(0.5, 0.5, "Waiting for loss data...", ha='center', va='center', color='gray')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.tight_layout(pad=1.0)
buf = io.BytesIO(); fig.savefig(buf, format='png', bbox_inches='tight'); plt.close(fig)

graph_widget = widgets.Image(value=buf.getvalue(), format='png', width=500, height=200)
stats_label = widgets.Label(value="Running...", layout=widgets.Layout(margin='0 0 0 15px', font_weight='bold', color='#FFD700'))

display_x0 = get_noise(1, 12, 128, device)
z_rk4 = ca_model.lift(display_x0)

def reset_latent():
    global display_x0, z_rk4
    display_x0 = get_noise(1, 12, 128, device)
    z_rk4 = ca_model.lift(display_x0)

def hard_reset_model(*args):
    global ca_model, optimizer, scaler, loss_total_log, loss_feature_log, loss_l2_log, loss_tv_log, best_loss, current_title

    ca_model = ClosedFormNCA(in_ch=12, latent_ch=48, kernel_size=5).to(device)
    optimizer = torch.optim.AdamW(ca_model.parameters(), lr=4e-3, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda')

    loss_total_log.clear()
    loss_feature_log.clear()
    loss_l2_log.clear()
    loss_tv_log.clear()
    best_loss = float('inf')
    current_title = f"Targeting {layer_dropdown.value} C{channel_slider.value}"
    reset_latent()

    fig, ax = plt.subplots(figsize=(6, 2.5))
    ax.text(0.5, 0.5, "Universe Destroyed. Rebuilding...", ha='center', va='center', color='red')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.tight_layout(pad=1.0)
    buf = io.BytesIO(); fig.savefig(buf, format='png', bbox_inches='tight'); plt.close(fig)
    graph_widget.value = buf.getvalue()
    stats_label.value = "⚡ Physics Re-rolled | 🏆 Loss: inf"

btn_restart_train.on_click(hard_reset_model)

def shader(t, dt):
    global z_rk4
    t_tensor = torch.tensor([t], device=device).view(1, 1, 1, 1)
    _, rgb_cf = ca_model(t_tensor, display_x0)

    sub_steps = max(1, int(dt / 0.1))
    for _ in range(sub_steps): z_rk4 = ca_model.step_latent_rk4(z_rk4, dt=dt/sub_steps)

    rgb_cf, rgb_rk4 = rgb_cf[0].permute(1, 2, 0), ca_model.project(z_rk4)[0].permute(1, 2, 0)
    return torch.cat([rgb_cf, rgb_rk4], dim=1)

# Build UI Layout
dor = InteractiveDOR(shader_func=shader, reset_cb=reset_latent)

# Style configuration and control grouping
controls_styled = widgets.HBox([
    layer_dropdown, channel_slider, dd_toggle, btn_restart_train
], layout=widgets.Layout(border='1px solid #333', padding='10px', margin='5px 0', border_radius='4px', background_color='#1a1a1a', align_items='center'))

layout_block = widgets.VBox([
    dor.ui,
    controls_styled,
    widgets.HBox([graph_widget, stats_label], layout=widgets.Layout(border='1px solid #333', padding='10px', border_radius='4px', background_color='#1a1a1a', align_items='center'))
], layout=widgets.Layout(padding='10px', background_color='#111', border_radius='8px'))

display(layout_block)

# --- Dual-Channel Browser-Driven Step Functions ---
last_render_time = 0.0
last_graph_update = 0.0
last_ui_update = 0.0
frames = deque(maxlen=30)
train_times = deque(maxlen=30)

# 1. CHANNEL A: Rendering step
def colab_render_step():
    global last_render_time, recorded_frames
    try:
        if not dor.is_running:
            return

        t_now = time.time()
        if not dor.is_paused:
            new_t = dor._pause_offset + (t_now - dor._unpause_wall) * 2.0
            dt = new_t - dor.t
            dor.t = new_t
        else:
            dt = 0.0

        with torch.no_grad():
            img_np = (shader(dor.t, dt).clamp(0, 1).mul_(255)).byte().cpu().numpy()

        # Push image data safely over WS
        dor.img_widget.value = cv2.imencode('.jpg', img_np[:, :, ::-1], [int(cv2.IMWRITE_JPEG_QUALITY), 75])[1].tobytes()

        # Append frame if recording is active
        if btn_record.value:
            recorded_frames.append(img_np.copy())
            if len(recorded_frames) >= 900: # Capped at 30 seconds to prevent RAM buffer leaks
                btn_record.value = False

        frames.append(t_now - last_render_time)
        dor.fps_label.value = f"FPS: {1.0/max(sum(frames)/len(frames), 1e-4):.0f} | t={dor.t:.2f}"
        last_render_time = t_now
    except Exception as e:
        print("\nRENDER ERROR:\n", traceback.format_exc())

# 2. CHANNEL B: Computational training step
def colab_train_step():
    global last_graph_update, last_ui_update, best_loss, current_title, train_times, loss_total_log, loss_feature_log, loss_l2_log, loss_tv_log
    try:
        if not dor.is_running or dor.is_paused:
            return

        t_now = time.time()
        t_start_step = time.time()

        t_train = torch.empty(2, 1, 1, 1, device=device).uniform_(4.0, 16.0)
        x0 = get_noise(2, 12, 128, device)

        with torch.amp.autocast('cuda'):
            _, rgb = ca_model(t_train, x0)
            views = random_crops(rgb, n=2)
            views = views + torch.randn_like(views) * 0.02

            _ = clip_model.encode_image(clip_norm(views))

            current_layer = layer_dropdown.value
            current_acts = _act[current_layer]

            if dd_toggle.value:
                fv_loss = -current_acts.square().mean() * 5.0
                current_title = f"Targeting {current_layer} (DeepDream)"
            else:
                target_ch = channel_slider.value
                fv_loss = -current_acts[:, target_ch].mean() * 5.0
                current_title = f"Targeting {current_layer} C{target_ch}"

            l2_reg = rgb.square().mean() * 0.1
            tv_loss = ((rgb[:,:,:,:-1] - rgb[:,:,:,1:]).square().mean() + (rgb[:,:,:-1,:] - rgb[:,:,1:,:]).square().mean()) * 0.005
            loss = fv_loss + l2_reg + tv_loss

        optimizer.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(ca_model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()

        current_loss = fv_loss.item()
        loss_total_log.append(loss.item())
        loss_feature_log.append(current_loss)
        loss_l2_log.append(l2_reg.item())
        loss_tv_log.append(tv_loss.item())

        if current_loss < best_loss: best_loss = current_loss
        train_times.append(time.time() - t_start_step)

        # Rate-limited UI text updates (~3 Hz)
        if t_now - last_ui_update >= 0.3:
            if len(train_times) > 0:
                avg_it_time = sum(train_times) / len(train_times)
                it_s = 1.0 / max(avg_it_time, 1e-4)
            else:
                it_s = 0.0
            stats_label.value = f"⚡ Train: {it_s:.0f} it/s | 🏆 Loss: {best_loss:.2f}"
            last_ui_update = t_now

        # Rate-limited multi-line chart image updates (~0.3 Hz)
        if len(loss_total_log) > 0 and (t_now - last_graph_update >= 3.0):
            fig, ax = plt.subplots(figsize=(6, 2.5))
            ax.plot(loss_total_log[-500:], color='#1f77b4', linewidth=1.5, label='Total Loss')
            ax.plot(loss_feature_log[-500:], color='#ff7f0e', linewidth=1.2, linestyle='--', label='Feature/Dream')
            ax.plot(loss_l2_log[-500:], color='#2ca02c', linewidth=1.0, alpha=0.7, label='L2 Reg')
            ax.plot(loss_tv_log[-500:], color='#d62728', linewidth=1.0, alpha=0.7, label='TV Reg')
            ax.legend(loc='upper right', fontsize=8, framealpha=0.5)
            ax.set_title(current_title, fontsize=10)
            ax.grid(True, linestyle='--', alpha=0.5); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            fig.tight_layout(pad=1.0)
            buf = io.BytesIO(); fig.savefig(buf, format='png', bbox_inches='tight'); plt.close(fig)
            graph_widget.value = buf.getvalue()
            last_graph_update = t_now
    except Exception as e:
         print("\nTRAIN ERROR:\n", traceback.format_exc())

# Interactive Mouse Painting / Damage Callback
def colab_perturb(x_norm, y_norm):
    global display_x0, z_rk4
    try:
        h, w = 128, 128
        grid_y = int(y_norm * h)

        # Grid visual coordinates are mapped 1:1 on both channels
        if x_norm < 0.5:
            # Clicked on left viewport (Closed-Form): perturb input initial state
            grid_x = int((x_norm * 2.0) * w)
            Y, X = torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device), indexing='ij')
            dist = (X - grid_x)**2 + (Y - grid_y)**2
            mask = (dist > 15**2).float().view(1, 1, h, w)  # Damage brush radius of 15 pixels
            display_x0 = display_x0 * mask
        else:
            # Clicked on right viewport (RK4): perturb active latent state
            grid_x = int(((x_norm - 0.5) * 2.0) * w)
            Y, X = torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device), indexing='ij')
            dist = (X - grid_x)**2 + (Y - grid_y)**2
            mask = (dist > 15**2).float().view(1, 1, h, w)
            z_rk4 = z_rk4 * mask
    except Exception as e:
        print("\nPerturbation Error:\n", traceback.format_exc())

# Register Python callbacks
output.register_callback('notebook.colab_render_step', colab_render_step)
output.register_callback('notebook.colab_train_step', colab_train_step)
output.register_callback('notebook.colab_perturb', colab_perturb)

# Output decoupled JavaScript controller with click event listeners and nearest neighbor zoom CSS
display(HTML("""
<style>
/* Dashboard Styling Override with nearest neighbor interpolation for sharp zooming */
.colab-nca-viewport img {
    cursor: crosshair !important;
    border: 1px solid #444;
    border-radius: 2px;
    image-rendering: pixelated !important;
    image-rendering: crisp-edges !important;
}
</style>

<script>
(async function() {
    // 1. Terminate previous loop iterations
    if (window.colab_render_interval) {
        clearInterval(window.colab_render_interval);
    }
    window.colab_train_active = false;
    await new Promise(resolve => setTimeout(resolve, 150)); // allow loops to wind down

    // Bind Mouse Drag Painting / Damage listeners to the viewport element
    const viewportContainer = document.querySelector('.colab-nca-viewport');
    if (viewportContainer) {
        const img = viewportContainer.querySelector('img');
        if (img) {
            let isDrawing = false;

            const handlePaint = (e) => {
                const rect = img.getBoundingClientRect();
                const x = (e.clientX - rect.left) / rect.width;
                const y = (e.clientY - rect.top) / rect.height;
                if (x >= 0 && x <= 1 && y >= 0 && y <= 1) {
                    google.colab.kernel.invokeFunction('notebook.colab_perturb', [x, y], {});
                }
            };

            img.addEventListener('mousedown', (e) => {
                isDrawing = true;
                handlePaint(e);
            });

            img.addEventListener('mousemove', (e) => {
                if (isDrawing) {
                    handlePaint(e);
                }
            });

            window.addEventListener('mouseup', () => {
                isDrawing = false;
            });

            console.log("Mouse paint event listeners attached to viewport.");
        }
    }

    window.colab_train_active = true;
    console.log("Dual channels activated.");

    // CHANNEL A: Lightweight rendering loop.
    // Fires every 33ms (30 FPS). We do NOT await it; it runs as an unblocked fire-and-forget signal.
    window.colab_render_interval = setInterval(() => {
        google.colab.kernel.invokeFunction('notebook.colab_render_step', [], {});
    }, 33);

    // CHANNEL B: Computational training loop.
    // Self-pacing: explicitly awaits Python completion to maximize GPU use without message queuing.
    while (window.colab_train_active) {
        try {
            await google.colab.kernel.invokeFunction('notebook.colab_train_step', [], {});
        } catch (e) {
            console.error("Train pipeline error:", e);
            await new Promise(resolve => setTimeout(resolve, 1000));
        }
        await new Promise(resolve => setTimeout(resolve, 2)); // micro-yield to keep DOM thread open
    }
})();
</script>
"""))